# Battery Thermal Surrogate: Data Exploration
Explore the generated thermal simulation dataset, visualize trajectories, material masks, and parameter distributions.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
from src.physics.materials import create_material_mask, compute_signed_distance
from src.physics.solver import HeatSolver2D

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
np.random.seed(42)
print("Setup complete!")

## 1. Generate Sample Data
Since we may not have a full dataset, let's generate a small sample using the physics solver.

In [ ]:
# Configuration
GRID_SIZE = 32
N_TRAJECTORIES = 5
N_STEPS = 100
DX = DY = 1e-3  # 1 mm grid spacing
T_AMB = 300.0   # Ambient temperature [K]

# Parameter ranges for random sampling
K_CELL_RANGE = (1.0, 5.0)       # Battery cell conductivity [W/(m*K)]
Q0_RANGE = (1e4, 1e5)           # Heat generation rate [W/m^3]
H_CONV_RANGE = (5.0, 50.0)      # Convective HTC [W/(m^2*K)]

# Create the material mask (shared across all trajectories)
mask = create_material_mask(grid_size=GRID_SIZE)
source_mask = (mask == 0).astype(np.float64)  # Heat generated only in battery cells

# Storage for trajectories and parameters
trajectories = []
params_list = []

for i in range(N_TRAJECTORIES):
    # Sample random physical parameters
    k_cell = np.random.uniform(*K_CELL_RANGE)
    q0 = np.random.uniform(*Q0_RANGE)
    h_conv = np.random.uniform(*H_CONV_RANGE)

    params = {"k_cell": k_cell, "q0": q0, "h_conv": h_conv}
    params_list.append(params)

    # Build per-material property arrays (indexed by material id)
    # 0=battery, 1=coolant (water), 2=insulation
    k_values = np.array([k_cell, 0.6, 0.04])
    rho_values = np.array([2500.0, 998.0, 30.0])
    cp_values = np.array([700.0, 4182.0, 1400.0])

    # Create solver from material fields
    solver = HeatSolver2D.from_material_fields(
        mask=mask,
        k_values=k_values,
        rho_values=rho_values,
        cp_values=cp_values,
        dx=DX, dy=DY,
        dt=solver_dt if 'solver_dt' in dir() else 0.0,  # placeholder
        T_amb=T_AMB,
        h_conv=h_conv,
    )

    # Use a stable time step
    solver_dt = solver.max_stable_dt * 0.5  # 50% safety margin
    solver = HeatSolver2D.from_material_fields(
        mask=mask,
        k_values=k_values,
        rho_values=rho_values,
        cp_values=cp_values,
        dx=DX, dy=DY,
        dt=solver_dt,
        T_amb=T_AMB,
        h_conv=h_conv,
    )

    # Initial condition: uniform ambient temperature
    T0 = np.full((GRID_SIZE, GRID_SIZE), T_AMB)

    # Run simulation
    traj = solver.solve(T0, n_steps=N_STEPS, q0=q0, source_mask=source_mask)
    trajectories.append(traj)

    print(f"Trajectory {i}: k_cell={k_cell:.2f}, q0={q0:.0f}, h_conv={h_conv:.1f}, "
          f"dt={solver_dt:.2e}, T_final=[{traj[-1].min():.1f}, {traj[-1].max():.1f}] K")

print(f"\nGenerated {N_TRAJECTORIES} trajectories, each with shape {trajectories[0].shape}")

## 2. Material Masks and Geometry
Visualize the material layout: battery cells, coolant channels, and insulation.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

# Discrete colormap for the three materials
material_names = {0: "Cell", 1: "Coolant", 2: "Insulation"}
colors = ["#e74c3c", "#3498db", "#95a5a6"]  # red, blue, grey
cmap_mask = ListedColormap(colors)
bounds = [-0.5, 0.5, 1.5, 2.5]
norm = BoundaryNorm(bounds, cmap_mask.N)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(mask, cmap=cmap_mask, norm=norm, origin="lower")
ax.set_title("Material Mask", fontsize=14)
ax.set_xlabel("x (grid cells)")
ax.set_ylabel("y (grid cells)")

# Build a legend
import matplotlib.patches as mpatches
legend_patches = [mpatches.Patch(color=colors[i], label=f"{i}: {material_names[i]}")
                  for i in range(3)]
ax.legend(handles=legend_patches, loc="upper right", fontsize=10)

plt.tight_layout()
plt.show()

## 3. Signed Distance Fields
SDFs encode distance to material interfaces, helping the model learn boundary effects.

In [ ]:
# Compute SDFs for battery cell and coolant regions
sdf_cell = compute_signed_distance(mask, material_id=0)
sdf_coolant = compute_signed_distance(mask, material_id=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(sdf_cell, cmap="RdBu_r", origin="lower")
axes[0].set_title("SDF: Battery Cell (id=0)", fontsize=12)
axes[0].set_xlabel("x (grid cells)")
axes[0].set_ylabel("y (grid cells)")
plt.colorbar(im0, ax=axes[0], label="Signed distance [cells]")

im1 = axes[1].imshow(sdf_coolant, cmap="RdBu_r", origin="lower")
axes[1].set_title("SDF: Coolant (id=1)", fontsize=12)
axes[1].set_xlabel("x (grid cells)")
axes[1].set_ylabel("y (grid cells)")
plt.colorbar(im1, ax=axes[1], label="Signed distance [cells]")

plt.tight_layout()
plt.show()

## 4. Temperature Trajectories
Visualize how temperature evolves over time for different parameter settings.

In [ ]:
# Show temperature snapshots for the first trajectory
traj = trajectories[0]
time_indices = [0, 25, 50, 75, 100]  # steps to visualize

fig, axes = plt.subplots(1, 5, figsize=(16, 3))

# Use consistent color range across all snapshots
vmin = traj.min()
vmax = traj.max()

for ax, t_idx in zip(axes, time_indices):
    im = ax.imshow(traj[t_idx], cmap="hot", origin="lower", vmin=vmin, vmax=vmax)
    ax.set_title(f"t = {t_idx} steps", fontsize=11)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

# Add a shared colorbar
fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label="Temperature [K]")

fig.suptitle(f"Temperature Evolution (k_cell={params_list[0]['k_cell']:.2f}, "
             f"q0={params_list[0]['q0']:.0f}, h_conv={params_list[0]['h_conv']:.1f})",
             fontsize=13, y=1.02)
plt.show()

## 5. Temperature Statistics
Analyze temperature distributions across the dataset.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Mean temperature over time for each trajectory ---
for i, traj in enumerate(trajectories):
    mean_temps = traj.mean(axis=(1, 2))
    axes[0].plot(mean_temps, label=f"Traj {i}")
axes[0].set_xlabel("Time step")
axes[0].set_ylabel("Mean Temperature [K]")
axes[0].set_title("Mean Temperature Over Time")
axes[0].legend(fontsize=8)

# --- Max temperature over time for each trajectory ---
for i, traj in enumerate(trajectories):
    max_temps = traj.max(axis=(1, 2))
    axes[1].plot(max_temps, label=f"Traj {i}")
axes[1].set_xlabel("Time step")
axes[1].set_ylabel("Max Temperature [K]")
axes[1].set_title("Max Temperature Over Time")
axes[1].legend(fontsize=8)

# --- Histogram of final temperatures across all pixels ---
final_temps = np.concatenate([traj[-1].flatten() for traj in trajectories])
axes[2].hist(final_temps, bins=50, edgecolor="black", alpha=0.7, color="#e74c3c")
axes[2].set_xlabel("Temperature [K]")
axes[2].set_ylabel("Pixel count")
axes[2].set_title("Final Temperature Distribution")
axes[2].axvline(final_temps.mean(), color="black", linestyle="--",
                label=f"Mean = {final_temps.mean():.1f} K")
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Final temperature stats across all trajectories:")
print(f"  Min:  {final_temps.min():.2f} K")
print(f"  Mean: {final_temps.mean():.2f} K")
print(f"  Max:  {final_temps.max():.2f} K")
print(f"  Std:  {final_temps.std():.2f} K")

## 6. Parameter Distributions
Examine the sampled physical parameters across trajectories.

In [ ]:
k_cells = [p["k_cell"] for p in params_list]
q0s = [p["q0"] for p in params_list]
h_convs = [p["h_conv"] for p in params_list]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.histplot(k_cells, bins=10, kde=False, ax=axes[0], color="#2ecc71", edgecolor="black")
axes[0].set_xlabel("k_cell [W/(m*K)]")
axes[0].set_ylabel("Count")
axes[0].set_title("Battery Cell Conductivity")
axes[0].axvline(np.mean(k_cells), color="black", linestyle="--",
                label=f"Mean = {np.mean(k_cells):.2f}")
axes[0].legend()

sns.histplot(q0s, bins=10, kde=False, ax=axes[1], color="#e67e22", edgecolor="black")
axes[1].set_xlabel("q0 [W/m^3]")
axes[1].set_ylabel("Count")
axes[1].set_title("Heat Generation Rate")
axes[1].axvline(np.mean(q0s), color="black", linestyle="--",
                label=f"Mean = {np.mean(q0s):.0f}")
axes[1].legend()

sns.histplot(h_convs, bins=10, kde=False, ax=axes[2], color="#3498db", edgecolor="black")
axes[2].set_xlabel("h_conv [W/(m^2*K)]")
axes[2].set_ylabel("Count")
axes[2].set_title("Convective HTC")
axes[2].axvline(np.mean(h_convs), color="black", linestyle="--",
                label=f"Mean = {np.mean(h_convs):.1f}")
axes[2].legend()

plt.tight_layout()
plt.show()

## 7. Model Input Channels
The neural network receives 9 input channels. Let's visualize each one.

In [ ]:
# Build the 9 input channels for the first trajectory at t=50
traj = trajectories[0]
p = params_list[0]

# Channel 0: Temperature field at t=50
T_snapshot = traj[50]

# Channels 1-3: Material masks (one-hot)
mask_cell = (mask == 0).astype(np.float64)
mask_coolant = (mask == 1).astype(np.float64)
mask_insulation = (mask == 2).astype(np.float64)

# Channels 4-6: Spatially-varying physical parameter fields
k_values = np.array([p["k_cell"], 0.6, 0.04])
k_field = k_values[mask].astype(np.float64)

q_field = p["q0"] * mask_cell  # Heat source only in battery cells

h_field = np.full_like(k_field, p["h_conv"])  # Uniform convection coefficient

# Channels 7-8: Signed distance fields
sdf_cell = compute_signed_distance(mask, material_id=0)
sdf_coolant = compute_signed_distance(mask, material_id=1)

# Assemble all channels
channels = [
    ("T (temperature)", T_snapshot, "hot"),
    ("Mask: Cell", mask_cell, "Reds"),
    ("Mask: Coolant", mask_coolant, "Blues"),
    ("Mask: Insulation", mask_insulation, "Greys"),
    ("k (conductivity)", k_field, "viridis"),
    ("q (heat source)", q_field, "YlOrRd"),
    ("h (convection)", h_field, "coolwarm"),
    ("SDF: Cell", sdf_cell, "RdBu_r"),
    ("SDF: Coolant", sdf_coolant, "RdBu_r"),
]

fig, axes = plt.subplots(3, 3, figsize=(12, 11))

for ax, (title, data, cmap) in zip(axes.flat, channels):
    im = ax.imshow(data, cmap=cmap, origin="lower")
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("9 Model Input Channels (t = 50 steps)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(f"Input tensor shape: ({len(channels)}, {GRID_SIZE}, {GRID_SIZE})")

## Summary
- Dataset contains 2D temperature trajectories from FD heat equation solver
- 3 material regions: battery cells, coolant channels, insulation
- 9 input channels encode temperature, geometry, and physical parameters
- Parameters sampled from realistic ranges for battery thermal management